In [0]:
PATH_PAYMENT_FILES = "/Volumes/customer_360/raw/source_files/landing_data/payments/"
PATH_PAYMENT_CHECKPOINTLOCATION_BRONZE = "/Volumes/customer_360/raw/source_files/checkpoints/payments/"
TABLE_BRONZE_PAYMENT = "customer_360.bronze.payments"
TABLE_METRIC = "customer_360.raw.stream_metrics"

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS customer_360.bronze.payments (
    payment_id STRING NOT NULL,
    order_id STRING NOT NULL,
    customer_id STRING NOT NULL,
    payment_method STRING,
    payment_status STRING,
    amount DECIMAL(12,2),
    payment_date TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA
""")

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DecimalType,
    TimestampType
)

payment_schema = StructType([
    StructField("payment_id", StringType(), False),
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("payment_method", StringType(), True),
    StructField("payment_status", StringType(), True),
    StructField("amount", DecimalType(12, 2), True),
    StructField("payment_date", TimestampType(), True),
    StructField("updated_at", TimestampType(), True)
])

In [0]:
payment_bronze=(
    spark
    .readStream
    .format("csv")
    .option("header",True)
    .schema(payment_schema)
    .load(PATH_PAYMENT_FILES)
)

In [0]:

query=(
    payment_bronze
    .writeStream
    .trigger(availableNow=True)
    .format("delta")
    .outputMode("append")
    .option("checkpointlocation",PATH_PAYMENT_CHECKPOINTLOCATION_BRONZE)
    .toTable(TABLE_BRONZE_PAYMENT)
)
query.awaitTermination()

In [0]:

import json
from pyspark.sql import Row
from datetime import datetime

metrics = []

for p in query.recentProgress:

    progress = json.loads(p.json)

    source = progress["sources"][0]

    metrics.append(
        Row(
            metric_time=datetime.now(),
            query_name="payment_bronze",
            batch_id=int(progress["batchId"]),
            input_rows=int(source.get("numInputRows", 0)),
            input_rows_per_second=float(source.get("inputRowsPerSecond", 0.0)),
            processed_rows_per_second=float(source.get("processedRowsPerSecond", 0.0)),
            processing_time_ms=int(
                progress.get("durationMs", {}).get("triggerExecution", 0)
            )
        )
    )

if metrics:
    metrics_df = spark.createDataFrame(metrics)

    metrics_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(TABLE_METRIC)
        

In [0]:
display(
    spark.sql(f"select * from {TABLE_BRONZE_PAYMENT}")

)